# TP4 (à compléter) : LLM & techniques de prompting

Remplacez chaque `...` et chaque `# TODO`. La **version complète** est le script
`../scripts/tp4_prompting.py` (à ne consulter qu'après avoir cherché).

**Objectif.** Écrire des prompts efficaces (**zero-shot**, **few-shot**,
**chain-of-thought**) et comparer les réponses de plusieurs LLM
(OpenAI / Gemini / Claude / modèle local).

**Mise en place (une fois), depuis le dossier `tp/` :**
1. `make install-llm` puis `cp .env.example .env`
2. Renseignez **au moins une** clé API dans `.env` (OpenAI, Gemini ou Claude), ou
   lancez un modèle local avec Ollama. **Ne mettez JAMAIS une clé dans le code.**

> Sans clé, vous pouvez quand même construire les prompts et les afficher ; seul
> l'appel au modèle nécessite une clé.

In [ ]:
# Cellule fournie : a executer telle quelle.
# Rend importable scripts/llm_clients.py quel que soit le dossier courant.
import os
import sys

for _p in ["../scripts", "scripts"]:
    if os.path.isdir(_p):
        sys.path.insert(0, _p)

from llm_clients import PROVIDERS, chat, disponible, modele

configures = [p for p in PROVIDERS if disponible(p)]
provider = configures[0] if configures else None
print("Fournisseurs configures :", configures or "aucun (voir .env.example)")
print("Fournisseur actif :", provider)


def demander(prompt, system=None, max_tokens=256):
    """Affiche le prompt, puis la reponse du fournisseur actif (si configure)."""
    if system:
        print("[system]", system)
    print("PROMPT >>>")
    print(prompt)
    if provider is None:
        print("\n(aucune cle : reponse non demandee - voir .env.example)")
        return
    print(f"\n[{provider} : {modele(provider)}] >>>")
    print(chat(provider, prompt, system=system, max_tokens=max_tokens))

## 1. Zero-shot
**Consigne.** Écrivez un prompt qui demande de classer l'avis en
**positif / négatif / neutre**, sans donner d'exemple.

In [ ]:
avis = "Le produit est arrive casse et le service client ne repond pas."
# TODO : prompt zero-shot (consigne + avis + "Reponse :")
prompt = ...
demander(prompt)

## 2. Few-shot
**Consigne.** Ajoutez **3 exemples étiquetés** avant la vraie question, et un
`system` qui impose **UN seul mot** en sortie. Testez ensuite un avis ambigu.

In [ ]:
system = ...   # TODO : role + format impose (un seul mot : positif/negatif/neutre)
prompt = (
    # TODO : 3 lignes "Avis : '...' -> label" puis un 4e avis SANS label
    ...
)
demander(prompt, system=system, max_tokens=16)

## 3. Chain-of-thought
**Consigne.** Comparez une formulation **directe** et une formulation
**« raisonne étape par étape »** sur le problème ci-dessous. (Bonne réponse : 11.)

In [ ]:
enonce = ("Un magasin recoit 3 cartons de 12 bouteilles. Il en casse 5, "
          "puis en vend 20. Combien de bouteilles reste-t-il ?")
prompt_direct = ...   # TODO : enonce + "Donne uniquement le nombre final."
prompt_cot = ...      # TODO : enonce + "Raisonne etape par etape, puis Resultat = <nombre>."

print("=== DIRECT ===")
demander(prompt_direct, max_tokens=32)
print("\n=== CHAIN-OF-THOUGHT ===")
demander(prompt_cot, max_tokens=256)

## 4. D'un prompt vague à un prompt précis
**Consigne.** Partez de `"Parle-moi du marketing"` (vague). Réécrivez-le en
précisant **rôle**, **tâche**, **format** (ex : 5 puces) et **longueur**. Comparez.

In [ ]:
prompt_vague = "Parle-moi du marketing."
prompt_precis = ...   # TODO : role + tache + format + longueur

print("=== VAGUE ===")
demander(prompt_vague, max_tokens=200)
print("\n=== PRECIS ===")
demander(prompt_precis, max_tokens=200)

## 5. Comparer les fournisseurs
**Consigne.** Envoyez un **même prompt** à tous les fournisseurs configurés et
comparez style, longueur et justesse.

In [ ]:
system = "Tu es un assistant concis. Reponds en une seule phrase."
prompt = "Explique en une phrase simple ce qu'est un LLM, pour un debutant."
for p in PROVIDERS:
    if disponible(p):
        print(f"\n[{p} : {modele(p)}]")
        # TODO : appeler chat(p, prompt, system=system, max_tokens=120) et l'afficher
        ...
    else:
        print(f"\n[{p}] non configure -> ignore")

## À rendre
- Pour chaque technique : le prompt final et la réponse obtenue.
- **Zero-shot vs few-shot** : le format est-il mieux respecté avec des exemples ?
- **Chain-of-thought** : le raisonnement demandé améliore-t-il la justesse
  (problème des bouteilles) ?
- **Vague vs précis** : ce que change l'ajout du rôle et du format.
- Comparaison d'**au moins deux fournisseurs** (style, longueur, justesse).

**Bonus.** Provoquez une **hallucination** (question piège), puis corrigez-la par
une consigne (ex : « réponds 'je ne sais pas' si tu n'es pas sûr »).